In [1]:
import pandas as pd

In [16]:
county = "harris"

In [17]:
income = pd.read_csv(f"../heat_maps/{county}_county/income_data.csv")
income["S1903_C03_001E"]

mobility = pd.read_csv(f"../heat_maps/{county}_county/vehicle_data.csv")
mobility["S0802_C01_094E"]

income = income.iloc[1:].reset_index(drop=True)
mobility = mobility.iloc[1:].reset_index(drop=True)

/tmp/ipykernel_2561202/2439008416.py:4: DtypeWarning: Columns (2,3,56,57,76,77,84,85,140,141,182,183,204,205,258,259,278,279,286,287,342,343,384,385,406,407,460,461,480,481,488,489,544,545,586,587,608,609,662,663,682,683,690,691,746,747,788,789,810,811,864,865,884,885,892,893,990,991) have mixed types. Specify dtype option on import or set low_memory=False.
  mobility = pd.read_csv(f"../heat_maps/{county}_county/vehicle_data.csv")


In [18]:
df = income.merge(mobility[['GEO_ID', 'S0802_C01_094E']], on='GEO_ID', how='inner')

df['S1903_C03_001E'] = pd.to_numeric(df['S1903_C03_001E'], errors='coerce')
df['S0802_C01_094E'] = pd.to_numeric(df['S0802_C01_094E'], errors='coerce')

In [19]:
df["income_percent"] = df["S1903_C03_001E"].rank(pct=True)
df["mobility_percent"] = 1 - df["S0802_C01_094E"].rank(pct=True)

df["raw_composite_rank"] = (df["income_percent"] + df["mobility_percent"]) / 2
df["final_income_mobility_composite"] = df["raw_composite_rank"].rank(pct=True)

df[['GEO_ID', 'S1903_C03_001E', "income_percent", 'S0802_C01_094E', "mobility_percent", "raw_composite_rank","final_income_mobility_composite"]].head()

,GEO_ID,S1903_C03_001E,income_percent,S0802_C01_094E,mobility_percent,raw_composite_rank,final_income_mobility_composite
0,1400000US48201100001,102588.0,0.790826,12.1,0.113063,0.451944,0.419266
1,1400000US48201210400,54004.0,0.320183,8.3,0.181081,0.250632,0.198165
2,1400000US48201210500,55337.0,0.342202,7.9,0.194595,0.268398,0.211927
3,1400000US48201210600,83618.0,0.655046,8.2,0.186036,0.420541,0.378899
4,1400000US48201210700,61161.0,0.394495,1.2,0.618468,0.506482,0.489908


In [20]:
df.to_csv(f"{county}_income_mobility_score.csv")